# Exercise 7 — Pre-Trained Generative Models and APIs

**CAS Agentic AI & Deep Learning - Computer Vision**

## Learning objectives
After this exercise you should be able to:
- Load and run pre-trained **Stable Diffusion** models for text-to-image generation
- Control image generation using **ControlNet** conditioning (e.g. Canny edges)
- Apply memory-efficiency techniques (fp16, VAE slicing, CPU offloading)
- Call external image-generation **APIs** (Replicate)

**Fill in code where you see `# YOUR CODE HERE`.**

<div style="background-color:#fff3cd;border-left:6px solid #ff9900;padding:14px 18px;margin:12px 0;border-radius:4px;color:#222">
<div style="font-size:18px;font-weight:bold">⚡ This exercise needs a GPU with ≥ 6 GB VRAM</div>
<div style="margin-top:8px">
<b>In Google Colab:</b> click <code>Runtime → Change runtime type → T4 GPU</code> (free tier, ≈ 15 GB VRAM).<br>
Stable Diffusion requires downloading ~4 GB of model weights. CPU inference is too slow to be practical.
</div>
</div>

## Setup Colab / Local Paths

If working in Colab: **Save the notebook to your personal Google Drive to persist model cache between sessions**.

In [ ]:
import sys
from pathlib import Path

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

IN_COLAB = "google.colab" in sys.modules
print(f"In Colab: {IN_COLAB}")

if IN_COLAB:
    print("Installing additional packages to Colab Environment")
    %pip install -q diffusers accelerate replicate

    ROOT_PATH = Path("/content")
    DATA_PATH = ROOT_PATH.joinpath("data")

else:
    import pyrootutils

    ROOT_PATH = pyrootutils.setup_root(
        search_from=".",
        indicator="pyproject.toml",
        project_root_env_var=True,
        dotenv=True,
        pythonpath=True,
        cwd=True,
    )

    DATA_PATH = ROOT_PATH.joinpath("data")

DATA_PATH.mkdir(parents=True, exist_ok=True)
HF_CACHE = DATA_PATH / "HF_CACHE"
HF_CACHE.mkdir(parents=True, exist_ok=True)
print(f"Data path: {DATA_PATH}")

Mount Google Drive if you want to persist the HuggingFace model cache (~4 GB for SD v1.5) across Colab sessions.

In [ ]:
def mount_drive(
    drive_mount_path: str = "/content/drive", data_path: str = "MyDrive/cas-dl-compvis"
) -> tuple[Path, Path]:
    from google.colab import drive

    root_path = Path(drive_mount_path)
    drive.mount(str(root_path))
    data_path = root_path.joinpath(data_path)
    return root_path, data_path


# Uncomment to persist model cache to Google Drive (recommended for repeated use):
# ROOT_PATH, DATA_PATH = mount_drive()

## Imports

In [ ]:
import io
import os
from getpass import getpass

import cv2
import numpy as np
import requests
import torch
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    StableDiffusionPipeline,
)
from matplotlib import pyplot as plt
from PIL import Image

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

if device.type == "cpu":
    raise RuntimeError(
        "No GPU detected. Stable Diffusion requires a GPU. "
        "In Colab: Runtime → Change runtime type → T4 GPU."
    )

## 1) Text-to-Image Generation

**Stable Diffusion** is a latent diffusion model that generates images from text prompts. We will use `runwayml/stable-diffusion-v1-5` — a widely-used open-source checkpoint.

First, log in to HuggingFace to access the model. You can create a free account at [huggingface.co](https://huggingface.co) and generate a token at *Settings → Access Tokens*.

In [ ]:
from huggingface_hub import login

HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN is None or HF_TOKEN == "":
    HF_TOKEN = getpass("Enter your HuggingFace token: ")
login(token=HF_TOKEN, add_to_git_credential=False)

Load the Stable Diffusion pipeline. The first run downloads ~4 GB of weights — subsequent runs use the local cache.

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    cache_dir=HF_CACHE,
)
pipe = pipe.to(device)
print("Pipeline loaded.")

### Generate your first image

In [ ]:
prompt = "A futuristic city floating in the clouds, artstation style"
generator = torch.Generator(device=device).manual_seed(123)
image = pipe(prompt, generator=generator).images[0]
image

### Task 1 — Explore Guidance Scale

The `guidance_scale` parameter controls how strongly the model follows the text prompt (classifier-free guidance). A higher value gives more prompt-aligned but less diverse images.

Generate the same prompt with `guidance_scale=1.0` (low) and `guidance_scale=12.0` (high). Display both side by side.

In [ ]:
generator_low = torch.Generator(device=device).manual_seed(123)
image_low = pipe(prompt, guidance_scale=1.0, generator=generator_low).images[0]

generator_high = torch.Generator(device=device).manual_seed(123)
image_high = pipe(prompt, guidance_scale=12.0, generator=generator_high).images[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image_low)
axes[0].set_title("guidance_scale=1.0")
axes[0].axis("off")
axes[1].imshow(image_high)
axes[1].set_title("guidance_scale=12.0")
axes[1].axis("off")
plt.tight_layout()

**Task**: Try your own creative prompt and describe what you observe.

In [ ]:
# YOUR PROMPT HERE — try something creative!
my_prompt = "..."

generator = torch.Generator(device=device).manual_seed(123)
image = pipe(my_prompt, generator=generator).images[0]
image

## 2) Conditional Image Generation with ControlNet

**ControlNet** extends diffusion models with spatial conditioning signals (e.g. Canny edges, depth maps, human pose). We use a ControlNet conditioned on Canny edge maps.

The conditioning image must share the spatial structure of the desired output — edges guide *where* the model places content, while the text prompt guides *what* it looks like.

### Step 1 — Extract Canny Edges from a Reference Image

In [ ]:
url = "https://github.com/pytorch/vision/blob/main/gallery/assets/dog2.jpg?raw=true"
r = requests.get(url, allow_redirects=True)
ref_image = Image.open(io.BytesIO(r.content)).convert("RGB")
ref_image

In [ ]:
# Convert to numpy, extract edges with Canny, then back to PIL
img_np = np.array(ref_image)
edges = cv2.Canny(img_np, threshold1=100, threshold2=200)
edges_rgb = np.stack([edges, edges, edges], axis=2)
canny_image = Image.fromarray(edges_rgb)
canny_image

### Step 2 — Load ControlNet + Stable Diffusion

In [ ]:
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny",
    torch_dtype=torch.float16,
    cache_dir=HF_CACHE,
)

pipe_cn = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    safety_checker=None,
    cache_dir=HF_CACHE,
).to(device)

pipe_cn.enable_attention_slicing()
print("ControlNet pipeline loaded.")

### Step 3 — Generate with Edge Conditioning

Generate an image guided by the Canny edge map and a text prompt.

In [ ]:
prompt_cn = "A dog lying on the grass, photorealistic, 8k resolution, highly detailed, cinematic lighting"
generator = torch.Generator(device=device).manual_seed(123)
output = pipe_cn(
    prompt_cn,
    image=canny_image.convert("RGB"),
    num_inference_steps=30,
    generator=generator,
)
output.images[0]

### Task 2 — Your Own Conditioning

1. Load any image of your choice (or use the reference image above).
2. Extract Canny edges with different `low_threshold` / `high_threshold` values.
3. Generate a new image using a prompt of your choice.

In [ ]:
# Load or reuse an image
my_ref = ref_image  # replace with your own image if desired

# YOUR CODE HERE: extract Canny edges with your threshold values
# YOUR CODE HERE: generate an image with your prompt using pipe_cn
raise NotImplementedError

## 3) Memory-Efficiency Techniques

Large diffusion models easily exceed 8–16 GB VRAM. `diffusers` supports several techniques to reduce memory usage:

| Technique | Method | Notes |
|---|---|---|
| Half precision | `torch_dtype=torch.float16` | ~2× VRAM reduction, minimal quality loss |
| VAE slicing | `pipe.enable_vae_slicing()` | Decode latents one at a time |
| Attention slicing | `pipe.enable_attention_slicing()` | Chunk attention computation |
| CPU offloading | `pipe.enable_model_cpu_offload()` | Move layers to RAM when not needed |

See the full guide: [HuggingFace — Reduce Memory Usage](https://huggingface.co/docs/diffusers/en/optimization/memory)

In [ ]:
pipe_mem = StableDiffusionPipeline.from_pretrained(
    "dreamlike-art/dreamlike-photoreal-2.0",
    torch_dtype=torch.float16,
    cache_dir=HF_CACHE,
).to(device)

# Enable memory-efficiency tricks
pipe_mem.enable_vae_slicing()
pipe_mem.enable_attention_slicing()

prompt_mem = (
    "photo, a church in the middle of a field of crops, bright cinematic lighting"
)
image_mem = pipe_mem(prompt_mem).images[0]
image_mem

## 4) Using APIs — Replicate

[Replicate](https://replicate.com) provides a REST API to hundreds of hosted ML models. You pay per inference run. This section demonstrates how to call Replicate models for image generation tasks.

Explore available models at [replicate.com/explore](https://replicate.com/explore).

You will need a Replicate account and API token.

In [ ]:
import replicate

REPLICATE_API_TOKEN = os.getenv("REPLICATE_API_TOKEN")
if REPLICATE_API_TOKEN is None or REPLICATE_API_TOKEN == "":
    REPLICATE_API_TOKEN = getpass("Enter your Replicate API key: ")
    os.environ["REPLICATE_API_TOKEN"] = REPLICATE_API_TOKEN

In [ ]:
def pil_to_bytes(pil_image, format="PNG"):
    buf = io.BytesIO()
    pil_image.save(buf, format=format)
    buf.seek(0)
    return buf

### Text-to-Image via Replicate (FLUX Schnell)

In [ ]:
output = replicate.run(
    "black-forest-labs/flux-schnell",
    input={"prompt": "A futuristic city floating in the clouds, artstation style"},
)
image_data = output[0].read()
image_flux = Image.open(io.BytesIO(image_data))
image_flux

### Image Upscaling

In [ ]:
url = "https://github.com/pytorch/vision/blob/main/gallery/assets/dog2.jpg?raw=true"
r = requests.get(url, allow_redirects=True)
lowres_image = Image.open(io.BytesIO(r.content)).convert("RGB")
lowres_image

In [ ]:
output = replicate.run(
    "recraft-ai/recraft-crisp-upscale",
    input={"image": pil_to_bytes(lowres_image)},
)
image_data = output.read()
image_upscaled = Image.open(io.BytesIO(image_data))
image_upscaled

### Task 3 — Explore a Model of Your Choice

Browse [replicate.com/explore](https://replicate.com/explore) and pick a model that interests you (e.g. style transfer, inpainting, face swap, video generation).

Call it via the Replicate API and display the result.

In [ ]:
# YOUR CODE HERE: call a Replicate model of your choice
# model_id = "..."
# output = replicate.run(model_id, input={...})
raise NotImplementedError